## Task 1: Preprocess and Explore the Data

### 1. Imports and Setup

This section imports all libraries needed for data extraction, cleaning,
visualization, and statistical testing:
- `yfinance` to pull historical price data
- `pandas` / `numpy` for data manipulation
- `matplotlib` / `seaborn` for visualization
- `statsmodels` for the Augmented Dickey-Fuller stationarity test

In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('deep')

### 2. Extract Historical Financial Data

We fetch daily OHLCV (Open, High, Low, Close, Volume) data for **TSLA**,
**BND**, and **SPY** from January 1, 2015 to June 30, 2026 using YFinance.

Each asset's data is downloaded separately and stored in a dictionary,
then combined into a single long-format DataFrame with a `Ticker` column
to identify which rows belong to which asset. This structure makes it
easy to group, filter, and compare assets later.

`try/except` blocks are used throughout so a failed download or
processing step for one ticker doesn't crash the entire pipeline.

In [7]:
tickers = ['TSLA', 'BND', 'SPY']
start_date = '2015-01-01'
end_date = '2026-06-30'

raw_data = {}

for ticker in tickers:
    try:
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
        if df.empty:
            raise ValueError(f"No data returned for {ticker}")
        raw_data[ticker] = df
        print(f"Downloaded {ticker}: {df.shape[0]} rows")
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

# Combine into a single long-format DataFrame with an asset identifier
combined_frames = []

for ticker, df in raw_data.items():
    try:
        temp = df.copy()
        temp['Ticker'] = ticker
        temp = temp.reset_index()
        combined_frames.append(temp)
    except Exception as e:
        print(f"Error processing {ticker} into combined frame: {e}")

try:
    combined_df = pd.concat(combined_frames, ignore_index=True)
    print(combined_df.head())
except Exception as e:
    print(f"Error concatenating dataframes: {e}")

[*********************100%***********************]  1 of 1 completed


Downloaded TSLA: 2888 rows


[*********************100%***********************]  1 of 1 completed


Downloaded BND: 2888 rows


[*********************100%***********************]  1 of 1 completed

Downloaded SPY: 2888 rows
Price        Date  Adj Close      Close       High        Low       Open  \
Ticker                  TSLA       TSLA       TSLA       TSLA       TSLA   
0      2015-01-02  14.620667  14.620667  14.883333  14.217333  14.858000   
1      2015-01-05  14.006000  14.006000  14.433333  13.810667  14.303333   
2      2015-01-06  14.085333  14.085333  14.280000  13.614000  14.004000   
3      2015-01-07  14.063333  14.063333  14.318667  13.985333  14.223333   
4      2015-01-08  14.041333  14.041333  14.253333  14.000667  14.187333   

Price       Volume Ticker Adj Close Close High Low Open Volume Adj Close  \
Ticker        TSLA              BND   BND  BND BND  BND    BND       SPY   
0       71466000.0   TSLA       NaN   NaN  NaN NaN  NaN    NaN       NaN   
1       80527500.0   TSLA       NaN   NaN  NaN NaN  NaN    NaN       NaN   
2       93928500.0   TSLA       NaN   NaN  NaN NaN  NaN    NaN       NaN   
3       44526000.0   TSLA       NaN   NaN  NaN NaN  NaN    Na

### 3. Data Cleaning and Understanding: Statistics & Data Types

Before modeling, we inspect the dataset's shape and distribution using
`.describe()` and check that each column has the correct data type.

- `Date` is converted to a proper `datetime` type (needed for time series
  operations and plotting)
- `Ticker` is converted to a `category` type for memory efficiency and
  cleaner grouping operations

In [8]:
try:
    print("Data types:\n", combined_df.dtypes)
    print("\nBasic statistics:\n", combined_df.describe())
except Exception as e:
    print(f"Error inspecting combined_df: {e}")

try:
    combined_df['Date'] = pd.to_datetime(combined_df['Date'])
    combined_df['Ticker'] = combined_df['Ticker'].astype('category')
except Exception as e:
    print(f"Error fixing dtypes: {e}")

Data types:
 Price      Ticker
Date                 datetime64[s]
Adj Close  TSLA            float64
Close      TSLA            float64
High       TSLA            float64
Low        TSLA            float64
Open       TSLA            float64
Volume     TSLA            float64
Ticker                         str
Adj Close  BND             float64
Close      BND             float64
High       BND             float64
Low        BND             float64
Open       BND             float64
Volume     BND             float64
Adj Close  SPY             float64
Close      SPY             float64
High       SPY             float64
Low        SPY             float64
Open       SPY             float64
Volume     SPY             float64
dtype: object

Basic statistics:
 Price                  Date    Adj Close        Close         High  \
Ticker                              TSLA         TSLA         TSLA   
count                  8664  2888.000000  2888.000000  2888.000000   
mean    2020-09-27 06:13: